<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">02 Demo - Measure First - Query Profiles and Benchmarks</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Demo - Measure First: Query Profiles and Benchmarks
#### Duration: 40 minutes

## Overview

Every optimization in this course starts with measurement. 

Before any layout change, join rewrite or compute swap, you will always open the same tools: the **Query Profile**, its **Performance Insights** and the `system_vw.query.history` system table. 
Then you'll be able to prove a change worked with an unbiased benchmark.

This part is the measurement foundation that every later part will assume. 

You'll learn to read a Query Profile end to end, let Performance Insights rank the bottleneck for you, find the worst offenders across the whole workspace from `system_vw.query.history` and run an unbiased before & after benchmark. 

You'll do all of this on a real, deliberately slow query over synthetic retail demo data.

> Serverless-first: this demo runs on a serverless notebook and the Query Profile is the native profiling tool.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Read a Query Profile** end to end: the operator tree, and the metrics that expose scanning, shuffle, spill, pruning and cache.
2. **Use Performance Insights** to rank a query's bottleneck and recognize the exact insight names (**DATA_SKEW**, **DATA_SPILL**, **EXPLODING_JOIN**).
3. **Query `system_vw.query.history`** to rank the worst offenders across the estate by duration, spill, shuffle and bytes read.
4. **Run an honest benchmark**: sidestep result reuse, warm up, report the median and track DBUs per run for A/B comparisons.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This course queries system tables through the <code>system_vw</code> catalog, a set of per-user views created by the <strong>Workspace-Setup</strong> notebook. 
    <br/><br/>
    In Vocareum this notebook is automatically executed. Each view reads the underlying <code>system.*</code> table and filters to the current user, so you see only your own activity. If you are in a brand new workspace, they are not available for 10-15 minutes and the cell can produce an error.<br/><br/>
    Two of these views lag behind live activity:
    <ul style="margin:8px 0 0;padding-left:20px;">
      <li><code>system_vw.query.history</code>: <strong>5 to 10 minutes</strong> lag. Run the query cells first so the rows land before you need them. If a cell returns no rows, wait a few minutes and re-run it.</li>
      <li><code>system_vw.billing.usage</code>: <strong>1 to 2 hours</strong> lag. For recent work this will likely still be empty. The demo falls back to a pre-built snapshot with the same schema.</li>
    </ul>
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **SQL and Python, and was tested using Serverless V5**. All purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It builds the retail demo tables and defines two queries: `BAD_QUERY_SQL` (the one we profile) and `GOOD_QUERY_SQL` (the A/B comparison in section F).

In [ ]:
%run ./Includes/Classroom-Setup-02

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.info { --accent:#1B5162; --tint:#F8F9FC; }
</style>

<div class="ds-callout info">
  <div class="ds-eyebrow">About the system_vw Catalog</div>
  <p style="margin:0 0 14px;">Throughout this course, system table queries use the <code>system_vw</code> catalog instead of <code>system</code> directly.</p>
  <table style="width:100%;border-collapse:collapse;font-size:14pt;margin-bottom:14px;">
    <thead>
      <tr style="border-bottom:2px solid #EEEDE9;text-align:left;">
        <th style="padding:8px 12px;">View</th>
        <th style="padding:8px 12px;">Source Table</th>
        <th style="padding:8px 12px;">Filter</th>
      </tr>
    </thead>
    <tbody>
      <tr style="border-bottom:1px solid #EEEDE9;">
        <td style="padding:8px 12px;"><code>system_vw.query.history</code></td>
        <td style="padding:8px 12px;"><code>system.query.history</code></td>
        <td style="padding:8px 12px;"><code>executed_by = current_user()</code></td>
      </tr>
      <tr style="border-bottom:1px solid #EEEDE9;background:#F9F7F4;">
        <td style="padding:8px 12px;"><code>system_vw.billing.usage</code></td>
        <td style="padding:8px 12px;"><code>system.billing.usage</code></td>
        <td style="padding:8px 12px;"><code>identity_metadata.run_as = current_user()</code></td>
      </tr>
      <tr style="border-bottom:1px solid #EEEDE9;">
        <td style="padding:8px 12px;"><code>system_vw.billing.list_prices</code></td>
        <td style="padding:8px 12px;"><code>system.billing.list_prices</code></td>
        <td style="padding:8px 12px;">None</td>
      </tr>
      <tr>
        <td style="padding:8px 12px;"><code>system_vw.storage.predictive_<wbr>optimization_<wbr>operations_history</code></td>
        <td style="padding:8px 12px;"><code>system.storage.predictive_<wbr>optimization_<wbr>operations_history</code></td>
        <td style="padding:8px 12px;">None</td>
      </tr>
    </tbody>
  </table>
  <p style="margin:0;">The raw <code>system.*</code> tables contain rows from every user in the workspace. The <code>system_vw</code> views pre-filter to <code>current_user()</code> where applicable, so your benchmark results, cost attribution and query history reflect only your own work.</p>
</div>
</div>

<div style="font-family:sans-serif; max-width:1100px; margin:0 auto; color:#0b2026; display:flex; flex-direction:column; gap:16px;">

<div style="display:flex; align-items:center; gap:20px; width:100%; border-radius:10px; padding:24px 28px; background:#1B5162; border-left:8px solid #FF5F46; box-shadow:0 2px 10px rgba(27,49,57,0.08);">
  <div style="width:60px; height:60px; border-radius:50%; flex-shrink:0; background:#FF5F46; color:#FFFFFF; font-size:30pt; font-weight:700; line-height:1; display:flex; align-items:center; justify-content:center;">!</div>
  <div>
    <div style="font-size:22pt; font-weight:700; color:#FFFFFF; line-height:1.2; margin-bottom:4px;">BAD_QUERY_SQL is deliberately slow</div>
  </div>
</div>

### The query we will profile
<br />
<div style="max-width:1100px;margin:0 auto;font-family:sans-serif;color:#0b2026;">
  <div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:12px 16px;border-radius:4px;font-size:14pt;line-height:1.55;margin-bottom:12px;">
    <strong>Reference only.</strong> This is the deliberately slow query. Read it now, then run it and read its profile.
  </div>
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">SELECT
  c.customer_id,
  c.tier,
  p.category,
  SUM(oi.quantity * p.list_price) AS revenue,
  COUNT(*) AS line_items
FROM orders o
JOIN order_items oi ON o.order_idx = oi.order_idx
JOIN products p    ON oi.product_idx = p.product_idx
JOIN customers c   ON o.customer_idx = c.customer_idx
WHERE date_format(o.order_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-12'
GROUP BY c.customer_id, c.tier, p.category
ORDER BY revenue DESC</pre>
</div>

### A1. Run the bad query, then open its profile

- In a notebook cell attached to a SQL warehouse or serverless, click **See performance** under the cell.
- Or go to **Query History**, click the query, and choose **See query profile**.

We materialize the result with `.count()` so the whole query runs.

In [ ]:
# .count() forces full execution so the Query Profile has something to show.
print("Running BAD_QUERY_SQL (open its Query Profile after this cell) ...")

_bad_df = spark.sql(BAD_QUERY_SQL)
_bad_rows = _bad_df.count()

print(f"Bad query returned {_bad_rows:,} grouped rows.")
print("Open the Query Profile: 'See performance' under this cell or View all in query history -> See query profile.")


##### EXPAND FOR ADDITIONAL DETAILS

<details>
<summary style="font-size:14pt; font-weight:600; color:#1B5162; cursor:pointer;">Why the profile shows shuffle hash joins (heavy shuffle) instead of broadcast</summary>

<div style="font-size:14pt; line-height:1.6; color:#0b2026; padding:12px 0;">

<p>The optimizer picks between two join strategies:</p>
<ul>
  <li><strong>Broadcast hash join</strong>: the smaller table is copied to every task. No network shuffle. Fast, but only works when the build side fits in memory (default threshold: 10 MB).</li>
  <li><strong>Shuffle hash join</strong>: both sides are redistributed (shuffled) by the join key. Works at any scale but produces visible <code>shuffle_read_bytes</code> and multiple Shuffle operators in the profile.</li>
</ul>

<p><strong>Why shuffle joins are guaranteed here:</strong> Our dimension tables are sized well above the broadcast threshold:</p>
<ul>
  <li><strong>customers</strong>: 5 M rows, \~126 MB on disk (12x the threshold)</li>
  <li><strong>products</strong>: 5 M rows, \~80 MB on disk (8x the threshold)</li>
</ul>

<p>At these sizes, the optimizer will never choose broadcast regardless of session state, ANALYZE stats, or AQE runtime estimates. Every join in this query uses shuffle hash join, which means both sides of each join are shuffled by the join key. Three shuffle hash joins produce six Shuffle operators, plus one more for the GROUP BY redistribution.</p>

<p><strong>What this means in the profile:</strong></p>
<ul>
  <li>High <code>shuffle_read_bytes</code> (over 1 GB of data moved across the network)</li>
  <li>Multiple Shuffle operators dominating the memory peak view</li>
  <li>Aggregated task time much larger than wall-clock (parallel shuffle work across many tasks)</li>
</ul>

<p><strong>In production:</strong> small dimension tables (under 10 MB projected) would be broadcast automatically. The shuffle you see here represents the common case of joining fact tables to large dimensions or other fact tables.</p>

</div>
</details>


## B. Measure First
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); display: flex; flex-direction: column; }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-head { display: flex; align-items: center; gap: 12px; margin-bottom: 6px; }
.ds-card-num { width: 34px; height: 34px; border-radius: 50%; background: var(--accent, #4299E0); color: #FFFFFF; font-size: 16pt; font-weight: 800; line-height: 1; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0; }
.ds-card-sub { font-size: 14pt; font-weight: 600; color: #1B5162; margin: 0 0 10px; line-height: 1.4; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.ds-viz { margin-top: auto; padding-top: 14px; border-top: 1px dashed #E0DED9; }
.ds-viz-cap { font-size: 14px; color: #618794; text-align: center; margin-top: 10px; line-height: 1.4; }
.ds-seg { height: 20px; border-radius: 3px; display: inline-block; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">Open the Query Profile before you change anything</div>
  </div>
</div>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-head"><div class="ds-card-num">1</div><div class="ds-card-title">Profile before optimizing</div></div>
    <div class="ds-card-sub">Find the real bottleneck, do not guess it.</div>
    <ul>
      <li>Engineers <strong>guess the wrong bottleneck</strong> often enough to make guessing a bad strategy</li>
      <li>A wrong guess <strong>wastes a sprint</strong> and can make performance worse</li>
      <li>The <strong>Query Profile</strong> names the actual bottleneck immediately</li>
    </ul>
    <div class="ds-viz">
      <div style="display:flex; height:20px; border-radius:4px; overflow:hidden; border:1px solid #EEEDE9;">
        <span class="ds-seg" style="width:22%; background:rgba(66,153,224,0.30);"></span>
        <span class="ds-seg" style="width:14%; background:rgba(66,153,224,0.30);"></span>
        <span class="ds-seg" style="width:44%; background:#98102A;"></span>
        <span class="ds-seg" style="width:20%; background:rgba(66,153,224,0.30);"></span>
      </div>
      <div class="ds-viz-cap">Query Profile: the <span style="color:#98102A;font-weight:700;">longest stage</span> is your bottleneck</div>
    </div>
  </div>

  <div class="ds-card coral">
    <div class="ds-card-head"><div class="ds-card-num">2</div><div class="ds-card-title">Confirm platform actions</div></div>
    <div class="ds-card-sub">See what the engine already handled.</div>
    <ul>
      <li><strong>Photon, AQE and Predictive Optimization</strong> automate most of what you used to tune by hand</li>
      <li>Measuring shows <strong>what engaged and what did not</strong></li>
      <li>Tune only the part <strong>the platform did not already cover</strong></li>
    </ul>
    <div class="ds-viz">
      <div style="display:flex; gap:6px; justify-content:center; flex-wrap:wrap;">
        <span style="font-size:13px; color:#0b2026; background:#FFFFFF; border:1px solid #FF5F46; border-radius:20px; padding:4px 10px;"><span style="color:#00A972;font-weight:700;">&#10003;</span> Photon</span>
        <span style="font-size:13px; color:#0b2026; background:#FFFFFF; border:1px solid #FF5F46; border-radius:20px; padding:4px 10px;"><span style="color:#00A972;font-weight:700;">&#10003;</span> AQE</span>
        <span style="font-size:13px; color:#0b2026; background:#FFFFFF; border:1px solid #C4CCD6; border-radius:20px; padding:4px 10px;"><span style="color:#618794;font-weight:700;">?</span> Predictive Opt</span>
      </div>
      <div class="ds-viz-cap">Confirm what <span style="color:#00A972;font-weight:700;">engaged</span> &#183; tune only the gap</div>
    </div>
  </div>

  <div class="ds-card amber">
    <div class="ds-card-head"><div class="ds-card-num">3</div><div class="ds-card-title">Prove the change worked</div></div>
    <div class="ds-card-sub">Benchmark honestly, or it is not proof.</div>
    <ul>
      <li>A faster-looking run is <strong>not proof</strong> because the <strong>result cache</strong> can serve the same answer for 24 hours</li>
      <li>An honest benchmark <strong>disables result reuse</strong>, warms up and reports a <strong>median</strong></li>
      <li>Track <strong>DBUs</strong> alongside duration.</li>
    </ul>
    <div class="ds-viz">
      <div style="display:flex; align-items:center; gap:10px; justify-content:center;">
        <div style="text-align:center;"><div style="width:44px; height:44px; background:rgba(152,16,42,0.15); border:1px solid #98102A; border-radius:4px;"></div><div style="font-size:12px; color:#618794; margin-top:3px;">before</div></div>
        <span style="color:#00A972; font-size:20px; font-weight:700;">&rarr;</span>
        <div style="text-align:center;"><div style="width:44px; height:22px; background:rgba(0,169,114,0.20); border:1px solid #00A972; border-radius:4px; margin-top:22px;"></div><div style="font-size:12px; color:#618794; margin-top:3px;">after</div></div>
      </div>
      <div class="ds-viz-cap">Median duration + DBUs, <span style="color:#0b2026;font-weight:700;">cache off</span></div>
    </div>
  </div>
</div>

<div style="border:2px solid #1B5162; border-radius:10px; background:#F8F9FC; padding:28px 32px 16px 32px;">
<div style="font-size:14pt; font-weight:700; color:#1B5162; margin-bottom:18px; letter-spacing:1px; text-transform:uppercase;">The optimization loop every part of this course follows</div>
<div style="display:grid; grid-template-columns:1fr auto 1fr auto 1fr auto 1fr; gap:0; align-items:stretch;">
  <div style="background:#FFFFFF; border:3px solid #4299E0; border-radius:10px; padding:16px 14px; box-shadow:0 2px 8px rgba(27,49,57,0.08); text-align:center;">
    <div style="font-size:14pt; font-weight:700; color:#4299E0; letter-spacing:1px; text-transform:uppercase; margin-bottom:6px;">Step 1</div>
    <div style="font-size:17pt; font-weight:700; color:#0b2026; margin-bottom:10px;">Measure</div>
    <div style="font-size:14pt; line-height:1.4; color:#0b2026;">Query Profile and system_vw.query.history</div>
  </div>
  <div style="display:flex; align-items:center; justify-content:center; color:#618794; font-size:26pt; line-height:1; padding:0 10px;">&rarr;</div>
  <div style="background:#FFFFFF; border:3px solid #00A972; border-radius:10px; padding:16px 14px; box-shadow:0 2px 8px rgba(27,49,57,0.08); text-align:center;">
    <div style="font-size:14pt; font-weight:700; color:#00A972; letter-spacing:1px; text-transform:uppercase; margin-bottom:6px;">Step 2</div>
    <div style="font-size:17pt; font-weight:700; color:#0b2026; margin-bottom:10px;">Name It</div>
    <div style="font-size:14pt; line-height:1.4; color:#0b2026;">Performance Insights ranks the bottleneck</div>
  </div>
  <div style="display:flex; align-items:center; justify-content:center; color:#618794; font-size:26pt; line-height:1; padding:0 10px;">&rarr;</div>
  <div style="background:#FFFFFF; border:3px solid #FF5F46; border-radius:10px; padding:16px 14px; box-shadow:0 2px 8px rgba(27,49,57,0.08); text-align:center;">
    <div style="font-size:14pt; font-weight:700; color:#FF5F46; letter-spacing:1px; text-transform:uppercase; margin-bottom:6px;">Step 3</div>
    <div style="font-size:17pt; font-weight:700; color:#0b2026; margin-bottom:10px;">Apply One Fix</div>
    <div style="font-size:14pt; line-height:1.4; color:#0b2026;">One change per iteration. Changing two things at once prevents you from identifying which one helped.</div>
  </div>
  <div style="display:flex; align-items:center; justify-content:center; color:#618794; font-size:26pt; line-height:1; padding:0 10px;">&rarr;</div>
  <div style="background:#FFFFFF; border:3px solid #FFAB00; border-radius:10px; padding:16px 14px; box-shadow:0 2px 8px rgba(27,49,57,0.08); text-align:center;">
    <div style="font-size:14pt; font-weight:700; color:#FFAB00; letter-spacing:1px; text-transform:uppercase; margin-bottom:6px;">Step 4</div>
    <div style="font-size:17pt; font-weight:700; color:#0b2026; margin-bottom:10px;">Benchmark</div>
    <div style="font-size:14pt; line-height:1.4; color:#0b2026;">Median duration and DBUs, no result cache</div>
  </div>
</div>
<div style="position:relative; height:80px; margin-top:12px;">
  <div style="position:absolute; left:37%; right:13%; top:0; height:52px; border-left:3px dashed #98102A; border-right:3px dashed #98102A; border-bottom:3px dashed #98102A; border-bottom-left-radius:14px; border-bottom-right-radius:14px;"></div>
  <div style="position:absolute; left:calc(37% - 7px); top:-6px; width:0; height:0; border-left:8px solid transparent; border-right:8px solid transparent; border-bottom:11px solid #98102A;"></div>
  <div style="position:absolute; left:50%; top:42px; transform:translateX(-50%); background:#F8F9FC; padding:2px 14px; color:#98102A; font-weight:700; font-size:14pt;">iterate</div>
</div>
</div>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why measure first
- Intuition about what is slow is wrong often enough to make it a costly strategy. Acting on a guess wastes a sprint and can regress performance.
- The platform automated a lot already: Photon, AQE, and Predictive Optimization handle compaction, partition tuning and join strategy.
- Measuring confirms what engaged and surfaces the gap it did not close.
- This part establishes the shared vocabulary every later part depends on. When Part 4 says "check the Shuffle in the profile" or Part 5 says "rank on `spilled_local_bytes`", you will already know exactly where to look.

### Reading the card visuals
- **Profile before optimizing:** the colored segmented strip at the bottom of Card 1 represents time per stage. Most segments are short; the long dark-red segment is the bottleneck stage the profile names for you.
- **Confirm platform actions:** the rounded badges in Card 2 (✓ Photon, ✓ AQE, ? Predictive Opt) show which automations engaged. Green check = active. Gray "?" = the one you still verify manually (and where any remaining tuning goes).
- **Prove the change worked:** the tall "before" box shrinking to a short "after" box in Card 3, measured with the result cache off and reported as a median with DBUs.

### The four-step loop
- **Measure**: Open the Query Profile for the target query or rank from `system_vw.query.history` to decide which query is worth the effort.
- **Name the bottleneck**: Performance Insights labels the top signal (DATA_SKEW, DATA_SPILL, EXPLODING_JOIN). A name picks the right fix from the course menu.
- **Apply one fix**: Two changes at once means you cannot isolate which one helped or hurt.
- **Benchmark honestly**: Sidestep result reuse (unique cache-busting query text), warm up with at least two runs, report the median and record DBUs per run.

</details>



## C. Query Profile Anatomy

Two things to know before you open a Query Profile:
<br />
<br />
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.tc { text-align: center; }
.ds-stat { font-size: 26pt; font-weight: 800; color: var(--accent); line-height: 1.1; }
.ds-stat-label { font-size: 14pt; color: #5E7077; margin-top: 6px; }
</style>

<div style="font-size:18pt;font-weight:700;color:#1B5162;margin-bottom:14px;">1. There are 2 clocks</div>
<div class="ds-row">
  <div class="ds-card blue tc">
    <div class="ds-stat">~6 s</div>
    <div class="ds-stat-label">Wall-clock duration</div>
    <div style="font-size:14pt;color:#0b2026;margin-top:10px;text-align:left;">What the user experienced. Three phases: <strong>scheduling</strong>, then <strong>optimization + pruning</strong>, then <strong>execution</strong>.</div>
  </div>
  <div class="ds-card coral tc">
    <div class="ds-stat">~70-150 s</div>
    <div class="ds-stat-label">Aggregated task time</div>
    <div style="font-size:14pt;color:#0b2026;margin-top:10px;text-align:left;">Total core-seconds across all tasks on all nodes. The work the cluster actually performed. (Lower when disk cache is warm.)</div>
  </div>
</div>
<div style="background:#F8F9FC;border:2px solid #EEEDE9;border-radius:8px;padding:14px 18px;margin-top:14px;font-size:14pt;color:#0b2026;">
  Aggregated task time is <strong>always at least</strong> wall-clock time. They are equal only in the worst case, when one task does all the work. What matters is how far apart they are:
  <ul style="margin:6px 0 0;padding-left:20px;">
    <li style="margin-bottom:8px;"><strong>Much longer than wall-clock</strong>: healthy parallelism. Many tasks ran at once across many cores (here, \~70-150 s of work finished in \~6 s).</li>
    <li style="margin-bottom:0;"><strong>Close to wall-clock</strong>: little parallelism. Work funneled through a few tasks, or back to the driver, so most cores sat idle. Waiting for a compute slot can also narrow the gap.</li>
  </ul>
</div>

<div style="font-size:18pt;font-weight:700;color:#1B5162;margin:24px 0 14px;">2. No profile for cache hits</div>
<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.12);padding:14px 18px;border-radius:4px;font-size:14pt;color:#0b2026;">
  A query served from the result cache shows <strong>Query profile is not available</strong>. 
  <br />
  <br />
  Run <code>SET use_cached_result = false</code> on a SQL warehouse, or add a trivial change to the query to force fresh execution on any compute. 
  <br /><br />
  Confirm a cache hit with <code>from_result_cache = true</code> in <code>system_vw.query.history</code>.
</div>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Where to find the two clocks
- **Left panel, Details tab**: shows both wall-clock duration (with its three-phase breakdown) and aggregated task time.
- The three wall-clock phases:
  - **Scheduling**: waiting for a compute slot. Long scheduling points at warehouse sizing or concurrency, not your query.
  - **Query optimization and file pruning**: the optimizer compiles the plan and applies file-level statistics. Long optimization points at a metadata or stats problem.
  - **Execution**: the actual work. This is what the DAG visualizes.

### The right panel (DAG)
- Read it bottom-up: Scans are at the leaves, the result is at the root.
- Three color modes (toggle buttons at the top):
  - **Time spent**: highlights the hot path. Start here.
  - **Memory peak**: highlights shuffle memory and spill risk.
  - **Rows**: highlights fan-out (row count jumps at joins) and collapse (row count drops at aggregates).
- Click any operator to open its detailed metrics in a side panel.
- The **Top operators** tab (left panel) ranks the costliest operators so you can jump straight to the bottleneck without traversing the DAG.

### Two caches, not one
- **Result cache** (serverless-only): workspace-shared, lasts about 24 hours, survives restarts. A cache-served query has **no Query Profile** because no execution happened. Disable with `SET use_cached_result = false`.
- **Disk cache** (local SSD): warms automatically on repeated reads. Shows as `read_io_cache_percent` in the profile. Does NOT prevent a Query Profile from appearing.
- Both can coexist. During benchmarking, disable result reuse but leave the disk cache alone (it reflects production behavior).

</details>


### C1. What to look for in this profile
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.danger { --bnr-bg: rgba(255,95,70,0.10); --bnr-accent: #FF5F46; --bnr-icon-fg: #FFFFFF; --bnr-title: #98102A; --bnr-tagline: #5E7077; }
/* --- clickable reveal --- */
.rc-hint { width: 100%; text-align: center; font-size: 14pt; color: #5E7077; font-style: italic; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: block; white-space: nowrap; font-size: 0; width: 100%; margin-top: 4px; }
.rc-box { display: inline-block; width: 32%; margin-right: 2%; min-height: 118px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 16px 14px; text-align: center; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); white-space: normal; }
.rc-box:last-child { margin-right: 0; }
.rc-box:hover { transform: translateY(-2px); }
.rc-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-label { display: block; font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; line-height: 1.2; pointer-events: none; }
.rc-title { display: block; font-size: 16pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-top: 6px; pointer-events: none; }
.rc-sub { display: block; font-size: 13px; color: #5E7077; line-height: 1.35; margin-top: 6px; pointer-events: none; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; width: 100%; }
.rc-detail-wrap.open { max-height: 900px; opacity: 1; margin-top: 12px; }
.rc-detail-card { background: #F9F7F4; border-radius: 10px; padding: 22px 26px; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 12px; line-height: 1.3; }
.rc-detail-body { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.rc-detail-body ul { margin: 0; padding-left: 22px; }
.rc-detail-body li { margin-bottom: 8px; }
.rc-detail-body code { font-family: monospace; font-size: 13pt; background: #EEEDE9; padding: 1px 5px; border-radius: 3px; color: #1B5162; }
.rc-viz { margin-top: 14px; padding-top: 12px; border-top: 1px dashed #E0DED9; }
.rc-viz-cap { font-size: 13px; color: #618794; text-align: center; margin-top: 8px; }
</style>

<div class="ds-banner danger">
  <div class="ds-banner-icon">!</div>
  <div>
    <div class="ds-banner-title">Name the bottleneck</div>
    <div class="ds-banner-tagline">This query was built to fail in three visible ways. Open the Query Profile and find each one.</div>
  </div>
</div>

<div class="rc-hint">Click each problem to see how to spot it in the profile.</div>

<div class="rc-row">
  <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#4299E0; border-top-color:#4299E0;">
    <span class="rc-label" style="color:#4299E0;">Read Less</span>
    <span class="rc-title">No pruning</span>
    <span class="rc-sub">Scan on orders: many files read, 0 pruned</span>
  </span><span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#00A972; border-top-color:#00A972;">
    <span class="rc-label" style="color:#00A972;">Shuffle Less</span>
    <span class="rc-title">Heavy shuffle</span>
    <span class="rc-sub">Shuffle operators with high peak memory</span>
  </span><span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#FFAB00; border-top-color:#FFAB00;">
    <span class="rc-label" style="color:#B27A00;">Skew</span>
    <span class="rc-title">Data skew</span>
    <span class="rc-sub">One hot task runs long after the rest finish</span>
  </span>
</div>

<div class="rc-detail-wrap" id="rc-detail-wrap">
  <div class="rc-detail-card" id="rc-detail-card"></div>
</div>

<div style="border-left:4px solid #4299E0;background:#F8F9FC;padding:14px 18px;border-radius:4px;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Your numbers may differ</strong>
  <div style="color:#0b2026;font-size:14pt;">The optimizer may choose a different execution plan between runs (different task counts, different operator ordering, varying parallelism). Exact GB, MB, seconds and operator IDs will vary. The <strong>patterns</strong> are stable: zero pruning, high shuffle memory, and the skewed task distribution.</div>
</div>

</div>

<script>
var REVEAL = [
  { title: "No pruning", color: "#4299E0",
    text: "<ul>"
      + "<li>A one-year filter on a two-year date range should prune <strong>roughly half the files</strong>. Here it <strong>prunes none</strong>.</li>"
      + "<li><code>date_format(order_date, 'yyyy-MM')</code> <strong>wraps the column in a function</strong>, so the engine cannot use per-file min/max statistics.</li>"
      + "<li>Look at the <strong>Scan</strong> operator: <strong>files read vs files pruned</strong> is the single most damning number.</li>"
      + "</ul>"
      + "<div class='rc-viz'>"
      + "<div style='display:flex;gap:4px;justify-content:center;'>"
      + "<span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span><span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span><span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span><span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span><span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span><span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span><span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span><span style='width:16px;height:20px;background:#4299E0;border-radius:2px;'></span>"
      + "</div>"
      + "<div class='rc-viz-cap'>All files read &#183; <span style='color:#98102A;font-weight:700;'>0 pruned</span> (all blue = all scanned)</div>"
      + "</div>" },
  { title: "Heavy shuffle", color: "#00A972",
    text: "<ul>"
      + "<li>Each shuffle hash join <strong>shuffles both sides</strong>; the GROUP BY adds one more. Color by <strong>Memory peak</strong> to see which shuffles dominate.</li>"
      + "<li>Shuffle is one of the <strong>top operators by aggregated task time</strong>.</li>"
      + "<li>High-cardinality <code>GROUP BY customer_id, tier, category</code> forces a <strong>full redistribution</strong> of all post-join rows before aggregation.</li>"
      + "</ul>"
      + "<div class='rc-viz'>"
      + "<div style='display:flex;align-items:center;justify-content:center;gap:12px;'>"
      + "<span style='display:flex;gap:6px;'>"
      + "<span style='display:flex;flex-direction:column;gap:2px;'><span style='width:14px;height:14px;background:#4299E0;border-radius:2px;'></span><span style='width:14px;height:14px;background:#FFAB00;border-radius:2px;'></span><span style='width:14px;height:14px;background:#98102A;border-radius:2px;'></span></span>"
      + "<span style='display:flex;flex-direction:column;gap:2px;'><span style='width:14px;height:14px;background:#98102A;border-radius:2px;'></span><span style='width:14px;height:14px;background:#4299E0;border-radius:2px;'></span><span style='width:14px;height:14px;background:#FFAB00;border-radius:2px;'></span></span>"
      + "</span>"
      + "<span style='color:#00A972;font-size:16px;font-weight:700;'>shuffle &#8594;</span>"
      + "<span style='display:flex;gap:6px;'>"
      + "<span style='display:flex;flex-direction:column;gap:2px;'><span style='width:14px;height:14px;background:#4299E0;border-radius:2px;'></span><span style='width:14px;height:14px;background:#4299E0;border-radius:2px;'></span></span>"
      + "<span style='display:flex;flex-direction:column;gap:2px;'><span style='width:14px;height:14px;background:#FFAB00;border-radius:2px;'></span><span style='width:14px;height:14px;background:#FFAB00;border-radius:2px;'></span></span>"
      + "<span style='display:flex;flex-direction:column;gap:2px;'><span style='width:14px;height:14px;background:#98102A;border-radius:2px;'></span><span style='width:14px;height:14px;background:#98102A;border-radius:2px;'></span></span>"
      + "</span>"
      + "</div>"
      + "<div class='rc-viz-cap'>Mixed keys &#8594; regrouped so each task holds one key (moved across the network)</div>"
      + "</div>" },
  { title: "Data skew", color: "#FFAB00",
    text: "<ul>"
      + "<li><strong>80% of orders belong to 50,000 whale customers</strong> (1% of 5 M). The shuffle concentrates their data onto a few tasks.</li>"
      + "<li>One task runs long after the rest finish. To confirm: <strong>click the hottest Shuffle node</strong> in the DAG, then check <strong>Max duration vs Median duration</strong> in the task metrics panel. A ratio of 5x or more confirms skew.</li>"
      + "<li><strong>Performance Insights</strong> may surface a <strong>DATA_SKEW</strong> label when the imbalance is severe enough. If it does not appear, the manual check above is definitive.</li>"
      + "<li>At demo scale, serverless prevents spill (<code>spilled_local_bytes = 0</code>). On classic compute or at production volumes, this becomes a <strong>DATA_SPILL</strong> insight.</li>"
      + "</ul>"
      + "<div class='rc-viz'>"
      + "<div style='display:flex;align-items:flex-end;justify-content:center;gap:7px;height:76px;'>"
      + "<span style='width:22px;height:14px;background:#FFAB00;border-radius:3px 3px 0 0;opacity:0.55;'></span>"
      + "<span style='width:22px;height:18px;background:#FFAB00;border-radius:3px 3px 0 0;opacity:0.55;'></span>"
      + "<span style='width:22px;height:12px;background:#FFAB00;border-radius:3px 3px 0 0;opacity:0.55;'></span>"
      + "<span style='width:22px;height:76px;background:#98102A;border-radius:3px 3px 0 0;'></span>"
      + "<span style='width:22px;height:16px;background:#FFAB00;border-radius:3px 3px 0 0;opacity:0.55;'></span>"
      + "<span style='width:22px;height:13px;background:#FFAB00;border-radius:3px 3px 0 0;opacity:0.55;'></span>"
      + "</div>"
      + "<div class='rc-viz-cap'>one <span style='color:#98102A;font-weight:700;'>hot task</span> &#183; the rest finish early</div>"
      + "</div>" }
];
var rcCurrent = null;
function rcSelect(id) {
  var wrap = document.getElementById('rc-detail-wrap');
  var card = document.getElementById('rc-detail-card');
  var d = REVEAL[id];
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  if (rcCurrent === id) { wrap.classList.remove('open'); rcCurrent = null; return; }
  rcCurrent = id;
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div class="rc-detail-title" style="color:' + d.color + ';">' + d.title + '</div><div class="rc-detail-body">' + d.text + '</div>';
  wrap.classList.add('open');
}
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Step-by-step profile walkthrough (serverless)

These steps describe what you see on **serverless compute with Photon**. Classic compute differences are noted at the end.

**1. Start with the top-level metrics (left panel, Details tab)**
- Compilation should be a small fraction of total time. If it is, this is an execution problem, not a planning problem.
- Rows read: \~75 M (across all four tables scanned). Rows produced: 1 (the `.count()` collapses everything).
- Files read vs files pruned: 96 files read, 0 pruned. A one-year filter on a two-year date range should prune roughly half. Here it prunes none.
- Aggregated task time (\~70-150 s) is much longer than wall-clock (\~6 s), confirming heavy parallel execution across \~160-260 tasks. The problem is volume-driven, not straggler-driven.

**2. Color the graph by Time spent**
- The graph shows multiple Shuffles, 4 Scans, 3 Inner Joins, and Grouping Aggregates. Each shuffle hash join shuffles both sides; the GROUP BY adds one more redistribution.
- On serverless, the disk cache warms almost immediately. Even on a first run, **Shuffle operators typically dominate time** because reads are served from fast local SSD. Grouping Aggregate is always in the top three.
- Grouping Aggregate is expensive because it collapses the full post-join fan-out, computing SUM and COUNT on every row.

**3. Color the graph by Memory peak**
- Several Shuffle operators in total, with a few dominating. Each shuffle hash join shuffles both sides, plus the GROUP BY redistribution.
- High memory in a Shuffle is a precursor to spill. At this demo scale, serverless allocates generous per-task memory so nothing spills (`spilled_local_bytes = 0`). The high peak memory is the observable leading indicator that WOULD spill on classic compute or at production volumes.

**4. Color the graph by Rows**
- The Scan on order_items shows 60 M rows (the full table). Its Shuffle also shows 60 M (all rows redistributed by join key).
- The Inner Join operator may report 30-60 M depending on which metric is shown (output rows vs probe-side input). The 60 M at Scan/Shuffle is the full order_items table being redistributed; the join output is ~30 M (~5 M filtered orders x ~6 items each).
- The date filter IS pushed to the Scan on orders as a row-level residual filter (it just cannot use file-level min/max stats for pruning). Orders are filtered to \~5 M during the scan, before the join.
- The optimizer inserts partial Grouping Aggregates to collapse rows before the next join.
- Scan output totals: order_items 60 M (full table) + orders \~5 M (after in-scan filter) + customers 5 M + products 5 M = \~75 M rows across all four scans.

**5. Open the Scan on orders specifically**
- The filter shown is `date_format(o.order_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-12'`.
- Because the column is wrapped in `date_format()`, the engine evaluates the function on every row rather than comparing file-level statistics. Files pruned = 0.
- Compare mentally: if the filter were `o.order_date >= '2024-01-01' AND o.order_date < '2025-01-01'`, the engine could prune files whose max(order_date) < '2024-01-01' or min(order_date) >= '2025-01-01'. That is what the GOOD query does.

**6. What you do NOT do here**
- You do not fix anything yet. The skill is naming the bottleneck from the profile.
- Parts 3 through 7 teach the fixes: layout (liquid clustering for pruning), shuffle reduction (pre-aggregation, fewer keys), and workload patterns (avoiding fan-out before aggregation).

**Classic compute differences (for reference)**
- On classic compute without a warm disk cache, the first run may show Scan operators dominating time (slow remote I/O). On serverless, the disk cache is warm almost immediately so this cold-read phase is rarely visible.
- Classic compute with fixed executor memory can show actual spill (`spilled_local_bytes > 0`) at this scale. Serverless allocates per-task memory proportional to data volume, preventing spill at demo scale.
- Task counts and aggregated task time may differ significantly (classic may show more tasks with higher total time) depending on cluster size and executor configuration.

</details>


## D. Query Performance Insights

**Query performance insights** (Now GA) - [AWS](https://docs.databricks.com/aws/en/sql/user/queries/performance-insights) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/user/queries/performance-insights) | [GCP](https://docs.databricks.com/gcp/en/sql/user/queries/performance-insights)

- When queries run, Databricks might return insights that identify opportunities to improve performance.
- When a query has actionable insights, an **Optimize** button appears that hands the query to **Genie Code** for a suggested rewrite.
- If your workspace doesn't have it on, you read the same signals directly from the Query Profile operators (section C2) and from `system_vw.query.history` (section E).

![Optimize Button](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/query-optimize.png)




<br></br>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
/* --- clickable reveal --- */
.rc-hint { width: 100%; text-align: center; font-size: 14pt; color: #5E7077; font-style: italic; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: block; white-space: nowrap; font-size: 0; width: 100%; margin-top: 4px; }
.rc-box { display: inline-block; width: 32%; margin-right: 2%; min-height: 130px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 16px 14px; text-align: center; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); white-space: normal; }
.rc-box:last-child { margin-right: 0; }
.rc-box:hover { transform: translateY(-2px); }
.rc-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-label { display: block; font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; line-height: 1.2; pointer-events: none; }
.rc-title { display: block; font-size: 16pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-top: 6px; pointer-events: none; }
.rc-sub { display: block; font-size: 13px; color: #5E7077; font-style: italic; line-height: 1.35; margin-top: 6px; pointer-events: none; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; width: 100%; }
.rc-detail-wrap.open { max-height: 900px; opacity: 1; margin-top: 12px; }
.rc-detail-card { background: #F9F7F4; border-radius: 10px; padding: 22px 26px; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 12px; line-height: 1.3; }
.rc-detail-body { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.rc-detail-body ul { margin: 0; padding-left: 22px; }
.rc-detail-body li { margin-bottom: 8px; }
.rc-detail-body code { font-family: monospace; font-size: 13pt; background: #EEEDE9; padding: 1px 5px; border-radius: 3px; color: #1B5162; }
.rc-viz { margin-top: 14px; padding-top: 12px; border-top: 1px dashed #E0DED9; }
.rc-viz-cap { font-size: 13px; color: #618794; text-align: center; margin-top: 8px; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">The platform can name the bottleneck for you</div>
    <div class="ds-banner-tagline">Performance Insights automatically detects, ranks and explains the costliest patterns. 
    <br /><br />
    The <b>Optimize</b> button hands the query to Genie Code for a suggested rewrite.</div>
  </div>
</div>

<div class="rc-hint">Click each insight to see how it shows up in this query.</div>

<div class="rc-row">
  <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#98102A; border-top-color:#98102A;">
    <span class="rc-label" style="color:#98102A;">Skew</span>
    <span class="rc-sub">"Data is distributed unevenly across computing resources."</span>
  </span><span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#4299E0; border-top-color:#4299E0;">
    <span class="rc-label" style="color:#4299E0;">Spill</span>
    <span class="rc-sub">"Data spilled to disk because it did not fit in memory."</span>
  </span><span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#00A972; border-top-color:#00A972;">
    <span class="rc-label" style="color:#00A972;">Stats</span>
    <span class="rc-sub">"Statistics can't be used because the filter converts the data type."</span>
  </span>
</div>

<div class="rc-detail-wrap" id="rc-detail-wrap">
  <div class="rc-detail-card" id="rc-detail-card"></div>
</div>


<div style="display:flex;align-items:center;gap:20px;width:100%;border-radius:10px;padding:24px 28px;background:rgba(0,169,114,0.10);border-left:8px solid #00A972;box-shadow:0 2px 10px rgba(27,49,57,0.08);margin-top:20px;">
  <div style="width:60px;height:60px;border-radius:50%;flex-shrink:0;background:#00A972;color:#FFFFFF;font-size:30pt;font-weight:700;line-height:1;display:flex;align-items:center;justify-content:center;">&#x2713;</div>
  <div>
    <div style="font-size:22pt;font-weight:700;color:#0b2026;line-height:1.2;margin-bottom:4px;">Genie Code recommendation</div>
    <div style="font-size:15pt;color:#5E7077;line-height:1.4;">Replace <code>date_format(order_date, 'yyyy-MM') BETWEEN ...</code> with a direct date range predicate, then cluster the table on <code>order_date</code> to enable file pruning.</div>
  </div>
</div>

<div style="margin-top:16px;">
  <div style="font-size:15pt;font-weight:700;color:#1B5162;margin-bottom:8px;">Step 1: Fix the predicate</div>
  <div style="font-size:14pt;color:#0b2026;margin-bottom:12px;line-height:1.55;">The engine sees a bare column compared to literals. It checks each file's min/max for <code>order_date</code> and skips files that fall entirely outside the range, instead of the column being wrapped in a function where the engine must open every file and evaluate every row.</div>
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">SELECT
  c.customer_id,
  c.tier,
  p.category,
  SUM(oi.quantity * p.list_price) AS revenue,
  COUNT(*) AS line_items
FROM orders o
JOIN order_items oi ON o.order_idx = oi.order_idx
JOIN products p ON oi.product_idx = p.product_idx
JOIN customers c ON o.customer_idx = c.customer_idx
WHERE o.order_date >= DATE'2024-01-01'
  AND o.order_date < DATE'2025-01-01'
GROUP BY c.customer_id, c.tier, p.category
ORDER BY revenue DESC</pre>
</div>

<div style="margin-top:16px;">
  <div style="font-size:15pt;font-weight:700;color:#1B5162;margin-bottom:8px;">Step 2: Fix the data layout</div>
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">ALTER TABLE orders SET TBLPROPERTIES ('delta.targetFileSize' = '16mb');
ALTER TABLE orders CLUSTER BY (order_date);
OPTIMIZE orders FULL;</pre>
</div>

<div style="border-left:4px solid #00A972;background:rgba(0,169,114,0.10);padding:14px 18px;border-radius:4px;margin:16px 0;">
  <strong style="display:block;color:#00A972;margin-bottom:6px;font-size:15pt;">Why two steps?</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;"><strong>Step 1</strong> removes the function wrapper so the engine CAN use per-file min/max statistics.
  <br /><br />
  <strong>Step 2</strong> reorders the data so files ARE organized by date, making pruning effective. Without both, <code>pruned_files</code> stays at 0.</div>
</div>

</div>

<script>
var REVEAL = [
  { title: "Skew", color: "#98102A",
    text: "<ul>"
      + "<li>A few tasks run much longer than the rest (<strong>stragglers</strong>).</li>"
      + "<li>Caused by <strong>whale keys, hot partitions, or uneven join key distribution</strong>.</li>"
      + "<li><strong>In our query</strong>: 50,000 whale customers (1% of 5 M) receive <strong>80% of orders</strong>, concentrating shuffle data onto a few tasks.</li>"
      + "<li>Color by <strong>Time</strong>: look for one task disproportionately slow in a Shuffle or Join operator.</li>"
      + "</ul>"
      + "<div class='rc-viz'>"
      + "<div style='display:flex;align-items:flex-end;justify-content:center;gap:7px;height:64px;'>"
      + "<span style='width:20px;height:14px;background:#98102A;border-radius:3px 3px 0 0;opacity:0.5;'></span>"
      + "<span style='width:20px;height:18px;background:#98102A;border-radius:3px 3px 0 0;opacity:0.5;'></span>"
      + "<span style='width:20px;height:12px;background:#98102A;border-radius:3px 3px 0 0;opacity:0.5;'></span>"
      + "<span style='width:20px;height:64px;background:#98102A;border-radius:3px 3px 0 0;'></span>"
      + "<span style='width:20px;height:15px;background:#98102A;border-radius:3px 3px 0 0;opacity:0.5;'></span>"
      + "</div>"
      + "<div class='rc-viz-cap'>one <span style='color:#98102A;font-weight:700;'>hot task</span> runs long &#183; the rest finish early</div>"
      + "</div>" },
  { title: "Spill", color: "#4299E0",
    text: "<ul>"
      + "<li>A task's <strong>working set exceeds its memory</strong> allocation, forcing a write to local disk.</li>"
      + "<li>Caused by <strong>too much data per task</strong>, often amplified by skew.</li>"
      + "<li><strong>In our query</strong>: six Shuffle operators with multi-GB peak memory. Serverless allocates generous memory, so <code>spilled_local_bytes = 0</code> at demo scale.</li>"
      + "<li>On classic compute with fixed executor memory, this pressure becomes <code>spilled_local_bytes > 0</code> and the DATA_SPILL insight fires.</li>"
      + "</ul>"
      + "<div class='rc-viz'>"
      + "<div style='display:flex;align-items:center;justify-content:center;gap:12px;'>"
      + "<span style='display:flex;flex-direction:column;align-items:center;gap:3px;'><span style='display:flex;flex-direction:column;gap:2px;'><span style='width:44px;height:11px;background:#4299E0;border-radius:2px;'></span><span style='width:44px;height:11px;background:#4299E0;border-radius:2px;'></span></span><span style='font-size:12px;color:#618794;'>memory</span></span>"
      + "<span style='color:#98102A;font-size:15px;font-weight:700;'>overflow &#8595;</span>"
      + "<span style='display:flex;flex-direction:column;align-items:center;gap:3px;'><span style='width:44px;height:22px;background:rgba(152,16,42,0.18);border:1px dashed #98102A;border-radius:2px;'></span><span style='font-size:12px;color:#618794;'>disk (spill)</span></span>"
      + "</div>"
      + "<div class='rc-viz-cap'>What does not fit in memory <span style='color:#98102A;font-weight:700;'>spills to slow disk</span></div>"
      + "</div>" },
  { title: "Stats", color: "#00A972",
    text: "<ul>"
      + "<li><strong>Delta data-skipping statistics exist but cannot be applied</strong> because the filter wraps the column in a function.</li>"
      + "<li>Status: <strong>Unused</strong>. The filter converts the data type, so per-file min/max stats are useless to the engine.</li>"
      + "<li><strong>In our query</strong>: <code>date_format(order_date, 'yyyy-MM')</code> wraps the column. The Scan on orders reads <strong>96 files, 0 pruned</strong>. A bare comparison would let the engine skip ~half.</li>"
      + "<li>Fix: rewrite as <code>order_date >= DATE'2024-01-01' AND order_date < DATE'2025-01-01'</code> so the engine can compare against file-level statistics.</li>"
      + "</ul>"
      + "<div class='rc-viz'>"
      + "<div style='display:flex;gap:4px;justify-content:center;flex-wrap:wrap;'>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "<span style='width:14px;height:18px;background:#00A972;border-radius:2px;'></span>"
      + "</div>"
      + "<div class='rc-viz-cap'>96 files scanned &#183; <span style='color:#98102A;font-weight:700;'>0 pruned</span> (stats exist but filter blocks their use)</div>"
      + "</div>" }
];
var rcCurrent = null;
function rcSelect(id) {
  var wrap = document.getElementById('rc-detail-wrap');
  var card = document.getElementById('rc-detail-card');
  var d = REVEAL[id];
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  if (rcCurrent === id) { wrap.classList.remove('open'); rcCurrent = null; return; }
  rcCurrent = id;
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div class="rc-detail-title" style="color:' + d.color + ';">' + d.title + '</div><div class="rc-detail-body">' + d.text + '</div>';
  wrap.classList.add('open');
}
</script>



##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Performance Insights
- Insights appear both in the query history list and inside the Query Profile. The query details panel shows a ranked summary; the full list is on the Performance Insights tab.
- The three this course leans on map to the workload patterns in Parts 3 through 7:
  - **Skew**: whale customers (50,000 out of 5 M) concentrate 80% of orders. The shuffle assigns their data disproportionately to a few tasks, creating stragglers. Parts 5 and 6 address skew with salting and AQE skew handling.
  - **Spill**: six Shuffle operators carry multi-GB peak memory. At demo scale, serverless absorbs it. On classic compute with fixed executor memory, these operators spill to disk. Part 6 (Compute) demonstrates this directly.
  - **Stats**: Delta data-skipping statistics exist on `orders` but the filter `date_format(order_date, 'yyyy-MM')` converts the column, making them unusable. Status: Unused. The Scan reads 96 files, 0 pruned. Fixing the predicate to a bare column comparison lets the engine apply per-file min/max stats and skip files outside the date range. This is the insight that directly motivates the Genie Code recommendation (Step 1).
- There is a larger family behind these three, including COVERAGE_PHOTON (query fell back off Photon), EXPLODING_JOIN (join output far exceeds both inputs), and SELECTIVE_JOIN (join produces far fewer rows than it reads). You do not need to memorize the full list.

### What Genie Code recommends
- The Optimize button analyzes the Query Profile and suggests a rewrite.
- For our bad query, the recommendation is two-part:
  1. **Fix the predicate**: replace `date_format(order_date, 'yyyy-MM') BETWEEN '2024-01' AND '2024-12'` with `o.order_date >= DATE'2024-01-01' AND o.order_date < DATE'2025-01-01'`. This removes the function wrapper so the engine can apply per-file min/max statistics.
  2. **Fix the data layout**: set `targetFileSize = '16mb'`, then `ALTER TABLE orders CLUSTER BY (order_date)`, then `OPTIMIZE FULL`. This reorders data into narrow date-band files so pruning is effective.
- The predicate fix alone does NOT deliver pruning improvement (as Section F proves with the A/B benchmark). Both steps are required.
- This course teaches the fundamentals first so you can judge whether Genie Code's rewrite is actually correct. We return to Genie Code in Part 8.
- Because Performance Insights is a Beta feature, no lesson depends exclusively on it. The Query Profile operators and `system_vw.query.history` give you the same facts without Insights enabled.

</details>



<div style="width: 100%; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <div style="margin-bottom: 10px;"><svg viewBox="0 0 806 200" fill="none" xmlns="http://www.w3.org/2000/svg" style="height:48px; width:auto; display:block;"><path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/><path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/><path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/><path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/><rect x="563.157" y="58.4648" width="242.843" height="84.9055" rx="42.4527" fill="#0B2026"/><path d="M664.499 114.947C661.815 114.947 659.493 114.366 657.532 113.205C655.596 112.018 654.1 110.38 653.042 108.289C652.01 106.173 651.493 103.735 651.493 100.974C651.493 98.187 652.01 95.7485 653.042 93.6584C654.1 91.5424 655.596 89.8909 657.532 88.7039C659.493 87.517 661.815 86.9235 664.499 86.9235C667.724 86.9235 670.356 87.7234 672.395 89.3232C674.433 90.8973 675.724 93.1294 676.266 96.0195H671.156C670.795 94.497 670.06 93.2971 668.95 92.4198C667.84 91.5424 666.344 91.1037 664.46 91.1037C662.757 91.1037 661.286 91.5037 660.048 92.3036C658.835 93.0778 657.893 94.2132 657.222 95.7098C656.577 97.1807 656.254 98.9353 656.254 100.974C656.254 103.012 656.577 104.767 657.222 106.238C657.893 107.683 658.835 108.805 660.048 109.605C661.286 110.38 662.757 110.767 664.46 110.767C666.344 110.767 667.84 110.367 668.95 109.567C670.06 108.741 670.795 107.606 671.156 106.161H676.266C675.749 108.896 674.459 111.05 672.395 112.625C670.356 114.173 667.724 114.947 664.499 114.947ZM697.803 114.947C695.196 114.947 692.887 114.366 690.874 113.205C688.887 112.018 687.326 110.38 686.191 108.289C685.055 106.173 684.488 103.722 684.488 100.935C684.488 98.1741 685.055 95.7485 686.191 93.6584C687.326 91.5424 688.887 89.8909 690.874 88.7039C692.887 87.517 695.196 86.9235 697.803 86.9235C700.46 86.9235 702.796 87.517 704.808 88.7039C706.821 89.8909 708.382 91.5424 709.492 93.6584C710.602 95.7485 711.156 98.1741 711.156 100.935C711.156 103.722 710.602 106.173 709.492 108.289C708.382 110.38 706.821 112.018 704.808 113.205C702.796 114.366 700.46 114.947 697.803 114.947ZM697.841 110.767C699.57 110.767 701.08 110.38 702.37 109.605C703.66 108.805 704.654 107.67 705.35 106.199C706.047 104.728 706.395 102.974 706.395 100.935C706.395 98.8966 706.047 97.1419 705.35 95.6711C704.654 94.2003 703.66 93.0778 702.37 92.3036C701.08 91.5037 699.57 91.1037 697.841 91.1037C696.112 91.1037 694.603 91.5037 693.313 92.3036C692.022 93.0778 691.016 94.2003 690.294 95.6711C689.597 97.1419 689.249 98.8966 689.249 100.935C689.249 102.974 689.597 104.728 690.294 106.199C691.016 107.67 692.022 108.805 693.313 109.605C694.603 110.38 696.112 110.767 697.841 110.767ZM720.339 114.482V87.3879H729.087C732.235 87.3879 734.829 87.9427 736.867 89.0523C738.931 90.1619 740.454 91.7359 741.434 93.7745C742.441 95.7872 742.944 98.187 742.944 100.974C742.944 103.709 742.441 106.096 741.434 108.135C740.454 110.147 738.931 111.708 736.867 112.818C734.829 113.928 732.235 114.482 729.087 114.482H720.339ZM724.984 110.534H728.855C731.203 110.534 733.048 110.16 734.39 109.412C735.757 108.638 736.725 107.541 737.293 106.122C737.886 104.677 738.183 102.961 738.183 100.974C738.183 98.9611 737.886 97.2452 737.293 95.8259C736.725 94.3809 735.757 93.2713 734.39 92.4972C733.048 91.6972 731.203 91.2973 728.855 91.2973H724.984V110.534ZM752.124 114.482V87.3879H769.619V91.1425H756.769V98.9224H768.458V102.6H756.769V110.728H769.619V114.482H752.124Z" fill="white"/><path d="M636.472 101.239L620.99 116.72L613.554 109.283L621.598 101.239L613.554 93.1943L620.99 85.7578L636.472 101.239Z" fill="#FF3621"/><path d="M584.879 101.241L600.36 85.7598L607.797 93.1968L599.753 101.241L607.797 109.286L600.36 116.723L584.879 101.241Z" fill="#FF3621"/></svg></div>
  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 14px;">Keep Building With Genie Code!</div>
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">If you have Performance Insights available, you can use <strong>Genie Code</strong> in the Query Profile (the Optimize button). The same assistant lives in notebooks and the SQL editor, so you can take it well beyond reading profiles. Ask it to rewrite a slow query, explain a Query Profile, or suggest clustering keys.</div>
  <a href="https://www.databricks.com/blog/introducing-genie-code" target="_blank" style="display: inline-block; background: #FF5F46; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">Introducing Genie Code &rarr;</a>
  <div style="display: flex; gap: 10px; margin-top: 12px;">
    <a href="https://docs.databricks.com/aws/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">AWS &rarr;</a>
    <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">Azure &rarr;</a>
    <a href="https://docs.databricks.com/gcp/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 24px; border-radius: 8px; text-decoration: none;">GCP &rarr;</a>
  </div>
</div>
</div>

## E. Find the Worst Offenders Across the Estate

A single profile tells you about one query. To find *which* queries to fix across the whole workspace, rank them from the `system_vw.query.history` system table. 

These columns are the triage dimensions:

- `total_duration_ms` for slowest overall (compilation plus execution, excludes result fetch)
- `spilled_local_bytes` for spill, `shuffle_read_bytes` for shuffle volume
- `read_bytes` and `read_files` versus `pruned_files` for scans that read too much
- `from_result_cache` to exclude cache hits, `read_io_cache_percent` for cold versus warm

In [ ]:
worst_offenders_sql = """
SELECT
    start_time,
    statement_text,
    total_duration_ms,
    execution_duration_ms,
    read_bytes,
    read_files,
    pruned_files,
    shuffle_read_bytes,
    spilled_local_bytes,
    from_result_cache,
    read_io_cache_percent
FROM system_vw.query.history
WHERE statement_type = 'SELECT'
    AND execution_status = 'FINISHED'
    AND from_result_cache = false
ORDER BY execution_duration_ms DESC
LIMIT 10
"""

display(spark.sql(worst_offenders_sql))

Queries that were run on serverless compute are captured in `system_vw.query.history`, but there is **ingestion latency** before rows appear.

The `statement_text` column returns **`<REDACTED>`** by default because SQL statements can contain PII (customer names, emails, addresses in WHERE clauses). Only **account admins** and members of the **`databricks_pii_access`** group see the full text. That group does not exist until an account admin creates it manually.

`statement_id` is always visible, never redacted, and uniquely identifies each query. 

1. Copy it from the Query Profile header (or from **Query History**), paste it into the cell below, then run the following cell to see the metrics for that exact run.

In [ ]:
# Paste the statement_id from the Query Profile header or Query History.
BAD_QUERY_STATEMENT_ID = "PASTE_YOUR_STATEMENT_ID_HERE"

In [ ]:
import time

if BAD_QUERY_STATEMENT_ID == "paste-your-statement-id-here":
    raise ValueError("Paste your statement_id into the cell above first, then re-run this cell.")

recent_sql = f"""
SELECT
    start_time,
    statement_id,
    total_duration_ms,
    execution_duration_ms,
    read_bytes,
    read_files,
    pruned_files,
    shuffle_read_bytes,
    spilled_local_bytes,
    from_result_cache,
    read_io_cache_percent
FROM system_vw.query.history
WHERE statement_id = '{BAD_QUERY_STATEMENT_ID}'
-- no-cache_{int(time.time())}
"""

df = spark.sql(recent_sql)

if df.count() == 0:
    print(f"No row yet for statement_id = '{BAD_QUERY_STATEMENT_ID}'.")
    print("system.query.history has 5–10 min ingestion lag. Wait and re-run.")
else:
    df.display()

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Reading the results column by column

**Timing columns**
- `total_duration_ms`: compilation plus execution (excludes result fetch). This is the number you rank on for "slowest queries."
- `execution_duration_ms`: execution only (no compilation). The gap between total and execution is compilation time. A large gap means the query plan was expensive to produce.

**Scan columns**
- `read_bytes`: bytes read after file pruning. If this is large relative to the table size, pruning failed.
- `read_files` vs `pruned_files`: the pruning ratio. Zero pruned files on a filtered query means the filter could not use file statistics (e.g. a function wraps the column).

**Shuffle and spill**
- `shuffle_read_bytes`: total bytes sent over the network for shuffle hash joins and redistributions. Maps to `networkSentBytes` in the Query Profile. Zero here means the optimizer used broadcast joins (small tables sent to every task in memory, no network shuffle).
- `spilled_local_bytes`: data written to disk because per-task memory was exhausted. Zero on serverless is expected (generous per-task allocation). Non-zero on classic compute signals memory pressure.

**Cache columns**
- `from_result_cache`: if `true`, the query was served from the result cache and did no real work. Always exclude these when triaging (they produce no Query Profile).
- `read_io_cache_percent`: percentage of `read_bytes` served from the local disk cache (SSD) rather than remote storage. Watch this warm across repeated runs: 0% (cold) to 60-70% (warm) correlates with a noticeable duration drop.

### Patterns visible in our results

**Disk cache warming**
- The first run after a table rebuild reads everything from remote storage (`read_io_cache_percent = 0`). Subsequent runs benefit from the disk cache (60%, then 71%). Duration drops accordingly. This is why the benchmarking recipe in Section F says "discard the warm-up run."

**Shuffle join confirmed**
- `shuffle_read_bytes` shows over 1 GB of data moved. This confirms the optimizer chose shuffle hash joins (both dimension tables at 5 M rows / 80-126 MB far exceed the 10 MB broadcast threshold).

**statement_text is redacted**
- The `statement_text` column returns `<REDACTED>` by default (only account admins and members of the `databricks_pii_access` group see the full text). You cannot use `LIKE` or `NOT LIKE` filters on it to identify or exclude specific queries. Use `statement_id` (always visible, never redacted) to look up individual runs, or use time windows to isolate benchmark phases.

### Fleet-wide triage
- The Query Profile is per-query. `system_vw.query.history` covers the whole estate (region-wide, all workspaces on the metastore), so it is where you decide which queries are worth your time.
- Rank on the symptom you care about: `total_duration_ms` for slowest, `spilled_local_bytes` for spill, `shuffle_read_bytes` for shuffle volume, `read_bytes` or `read_files` vs `pruned_files` for scans that read too much.
- Always filter `from_result_cache = false` when hunting for slow work. A cached result is fast for the wrong reason.
- To open a profile from a row here, copy its `statement_id` into the Statement ID filter in Query History.
- Access is admin-restricted by default. If you cannot read it, an admin can grant a dynamic view. The identical per-query numbers still live in each Query Profile.
- There is **ingestion latency** of 5 to 10 minutes before new rows appear. If a query you just ran is missing, wait and re-query.

</details>


## F. A / B Benchmarking
<br />
<div style="border-left:6px solid #FF5F46;background:#FFF6F4;border-radius:8px;padding:16px 20px;margin:0 auto 18px;max-width:1100px;font-family:sans-serif;font-size:14pt;line-height:1.55;color:#0b2026;">
<strong style="color:#98102A;">Serverless is not deterministic.</strong>
<br /><br />
Compute, memory and CPU assignment vary from run to run, so two runs of the same query rarely post identical times. 
<br /><br />
This is expected, which is why a single measurement may not prove the performance optimization. 
<br /><br />
Disable result reuse, discard the warm-up run and report the <strong>median</strong> across several runs and judge a change by the query plan and the task-time trend.
</div>
<div class="acc-wrap">
<style>
.acc-wrap { max-width: 1100px; margin: 0 auto; font-family: sans-serif; }
.acc-wrap * { box-sizing: border-box; }
.acc-item { border: 2px solid #EEEDE9; border-left: 8px solid var(--accent, #1B5162); border-radius: 8px; margin-bottom: 10px; overflow: hidden; transition: border-color 0.2s; }
.acc-item.open { border-color: var(--accent); border-left-color: var(--accent); }
.acc-header { display: flex; align-items: center; gap: 14px; padding: 16px 18px; cursor: pointer; user-select: none; background: #F9F7F4; transition: background 0.15s; }
.acc-item.open .acc-header { background: var(--tint); }
.acc-num { width: 36px; height: 36px; border-radius: 50%; background: var(--accent); color: #fff; font-size: 16pt; font-weight: 800; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
.acc-summary { flex: 1; font-size: 16pt; font-weight: 700; color: #0b2026; }
.acc-chevron { font-size: 14pt; color: var(--accent); transition: transform 0.25s; }
.acc-item.open .acc-chevron { transform: rotate(180deg); }
.acc-body { max-height: 0; opacity: 0; overflow: hidden; transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding: 0 18px 0 68px; font-size: 14pt; line-height: 1.6; color: #0b2026; }
.acc-item.open .acc-body { max-height: 700px; opacity: 1; padding: 4px 18px 18px 68px; }
.acc-body ul { margin: 8px 0 0; padding-left: 20px; }
.acc-body li { margin-bottom: 8px; }
.acc-body code { font-family: monospace; font-size: 13pt; background: #EEEDE9; padding: 1px 5px; border-radius: 3px; color: #1B5162; }
.acc-fix { margin-top: 12px; padding: 10px 14px; border-left: 4px solid var(--accent); background: var(--tint); border-radius: 0 6px 6px 0; font-size: 14pt; line-height: 1.55; }
</style>

<div id="acc-container"></div>

</div>

<script>
var ACC_DATA = [
  { accent: '#4299E0', tint: 'rgba(66,153,224,0.10)', summary: 'Turn result reuse off',
    detail: '<ul>'
      + '<li>On a <strong>SQL warehouse</strong>: run <code>SET use_cached_result = false</code> so a repeat run re-executes instead of returning the cached result.</li>'
      + '<li>On <strong>serverless notebook compute</strong> (Spark Connect): that config is not available. Instead, append a unique comment to each run (for example <code>-- run_1_1690000000</code>). This changes the query text hash so the result cache cannot match.</li>'
      + '<li>Both approaches disable the result cache only. The disk cache is separate, which is why step 2 matters.</li>'
      + '</ul>',
    fix: 'You are measuring the query, not the cache. A cache hit produces no Query Profile to read.' },
  { accent: '#00A972', tint: 'rgba(0,169,114,0.10)', summary: 'Warm up, then measure steady state',
    detail: '<ul>'
      + '<li>Discard the first run. It pays cold disk-cache and compilation costs.</li>'
      + '<li>Then run several times and report the <strong>median</strong>, not the best or the average.</li>'
      + '</ul>',
    fix: 'The median resists the odd slow outlier and reflects what users actually get.' },
  { accent: '#FFAB00', tint: 'rgba(255,171,0,0.12)', summary: 'Track DBUs per run, not just time',
    detail: '<ul>'
      + '<li>Cost is part of the score. A faster query that costs more is not always a win.</li>'
      + '<li>Immediate proxy: <code>total_task_duration_ms</code> from <code>system_vw.query.history</code>. On serverless, DBUs scale directly with aggregated compute seconds.</li>'
      + '<li>Full attribution: <code>system_vw.billing.usage</code> (1 to 2 hour ingestion lag, 10 to 60 min granularity). Isolate your compute over a known window and prorate.</li>'
      + '</ul>',
    fix: 'Report duration AND cost per run side by side.' },
  { accent: '#FF5F46', tint: 'rgba(255,95,70,0.10)', summary: 'Compare A vs B fairly',
    detail: '<ul>'
      + '<li>Change one thing. Same warehouse, same data, no competing queries.</li>'
      + '<li>Confirm fairness in <code>system_vw.query.history</code>: <code>from_result_cache = false</code> on both.</li>'
      + '<li>If the clock says B won but the profile shows identical metrics, the "win" is <strong>disk cache</strong> related (A ran cold, B ran warm), not necessarily a real improvement.</li>'
      + '</ul>',
    fix: 'Below we test twice: <strong>Phase 1</strong> (predicate fix only, no layout change) exposes disk cache related improvement. <strong>Phase 2</strong> (after clustering) shows the real improvement backed by profile proof.' }
];

(function() {
  var container = document.getElementById('acc-container');
  ACC_DATA.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'acc-item';
    div.style.setProperty('--accent', item.accent);
    div.style.setProperty('--tint', item.tint);
    div.innerHTML =
      '<div class="acc-header" onclick="toggleAcc(' + i + ')">'
      + '<div class="acc-num">' + (i + 1) + '</div>'
      + '<div class="acc-summary">' + item.summary + '</div>'
      + '<div class="acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="acc-body">'
      + '<div>' + item.detail + '</div>'
      + '<div class="acc-fix">' + item.fix + '</div>'
      + '</div>';
    container.appendChild(div);
  });
})();

function toggleAcc(idx) {
  var items = document.querySelectorAll('.acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The benchmarking recipe
- Step 1, bypass the result cache. Databricks caches results by default, so a second run of the same text returns instantly and teaches you nothing about the query. On a SQL warehouse, `SET use_cached_result = false` disables this. On serverless notebook compute (Spark Connect), that config is not available, so you append a unique comment to each SQL text (changing the hash the cache keys on). Either way, you are disabling the result cache only, not the disk cache.
- Step 2, warm-up plus median. The first run pays cold disk-cache reads and query compilation. Throw it away, then take the median of the next several runs. Median over mean because one stray slow run should not dominate.
- Step 3, DBUs per run. Duration alone hides cost. The immediate proxy is `total_task_duration_ms` from `system_vw.query.history`, which measures aggregated compute seconds across all tasks (directly proportional to DBUs on serverless). For full dollar attribution, query `system_vw.billing.usage` (1 to 2 hour ingestion lag, 10 to 60 min window granularity).
- Step 4, fair A/B. Change exactly one variable, run both on the same warehouse with nothing else competing, and verify neither run was a cache hit. Then compare medians and DBU per run. Crucially, if the clock says B is faster but the profile shows identical bytes, files, and task time, you have a disk cache artifact, not a real win. We demonstrate this below in two phases: Phase 1 (predicate fix only) shows the artifact; Phase 2 (after clustering) shows the genuine improvement confirmed by profile metrics.

</details>


### F1. Two-phase benchmark

Genie Code recommended two fixes (section D):
1. **Fix the predicate**: replace `date_format(order_date, 'yyyy-MM') BETWEEN ...` with a direct date range (sargeable, as defined in section D).
2. **Fix the data layout**: `ALTER TABLE orders SET TBLPROPERTIES ('delta.targetFileSize' = '16mb')`, then `ALTER TABLE orders CLUSTER BY (order_date)`, then `OPTIMIZE orders FULL`.

We benchmark **twice** to prove both steps are needed:
- **Phase 1** (below): compare A (bad predicate) vs B (sargeable predicate) on the **original unclustered** data. No clustering yet, so every file spans the full date range and B cannot prune.
- **Phase 2** (after applying Step 2): re-run B on **clustered** data. Now files have narrow date bands and the sargeable predicate prunes files outside the filter range.

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why the comparison is fair: ANALYZE TABLE at setup

The classroom setup runs `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS` on all four tables before any benchmark query executes. This gives the optimizer full column statistics from the start.

**Without ANALYZE**, the first query execution "teaches" AQE runtime sizes. Subsequent queries in the same session may use a different join strategy (e.g., broadcast instead of shuffle hash) because AQE now knows the projected output is small enough to broadcast. This makes cross-phase comparisons unfair: Phase 1 A could use shuffle hash (cold) while Phase 2 B uses broadcast (warm AQE), or vice versa.

**With ANALYZE**, the optimizer already knows every table's row count, column cardinality, and size estimate before any query runs. It makes the same join strategy choice in both phases. The only variable between phases is the file layout (clustering + OPTIMIZE). This isolates the effect of the layout fix and makes `total_task_duration_ms` a fair comparison.

If you ever see wildly different `shuffle_read_bytes` between phases (e.g., 1.2 GB in one phase and 0 in the other), it means the optimizer chose different join strategies. Re-running the setup (which re-ANALYZEs) resets the session.

</details>

In [ ]:
run_benchmark()

### F1a. Phase 1 result

The wall clock sometimes is faster sometimes not. But **the layout hasn't changed**, so neither query benefits from clustering.

Any "speedup" is a disk cache artifact: A runs first (cold), B runs second (warm).

The profile will confirm almost identical metrics for both.

Now apply Step 2: set `targetFileSize = '16mb'` (so OPTIMIZE produces enough files for pruning to be visible), cluster on `order_date`, and `OPTIMIZE FULL`. 

This physically sorts data by date and splits the table into 4-5 narrow date-band files. 

The sargeable predicate in B can then skip files whose date ranges fall entirely outside the filter.

In [ ]:
apply_layout_fix()

In [ ]:
# Phase 2: re-run B (tuned query) now that clustering is in place.
# ANALYZE TABLE ran at setup so the optimizer has stable stats across both phases. 
# The join strategy stays the same and the only change is the file layout.

from datetime import datetime, timezone

_phase2_start = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
median_b_after = benchmark_query(GOOD_QUERY_SQL, "B (after clustering)")
_phase2_end = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")

print(f"\n{'='*60}")
print(f"Phase 1 A (bad query, no clustering): {median_a:.2f}s")
print(f"Phase 2 B (fixed predicate + clustering): {median_b_after:.2f}s")
print(f"Speedup: {median_a / median_b_after:.2f}x")
print(f"\nBenchmark window: {_phase2_start} to {_phase2_end} UTC")


<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Next</div>
  <div class="ds-callout-title">Comparing Phase 1 A (no clustering) vs Phase 2 B (after clustering)</div>
  <ul>
    <li>The cell below queries <code>system_vw.query.history</code> for the median profile metrics from both benchmark phases: A (the bad query on the original layout) vs B (the tuned query after clustering).</li>
    <li><strong>Ingestion lag:</strong> <code>system_vw.query.history</code> has a 5 to 10 minute delay. If the result is empty, wait a few minutes and re-run the cell.</li>
  </ul>
</div>
</div>

In [ ]:
prove_benchmark()

These are the columns from `system_vw.query.history` (median across 5 measured runs):

**Columns:**
- `execution_duration_ms`: wall-clock time. Useful for latency, but inflated by cache warming.
- `total_task_duration_ms`: aggregated compute across all tasks. The DBU proxy on serverless. Same value = same cost, regardless of wall-clock.
- `read_bytes` / `read_files` / `pruned_files`: how much data the scan touched and how much it skipped. Pruning only fires when the predicate is sargeable AND the file layout supports it.
- `shuffle_read_bytes`: data moved between tasks for joins and aggregations. Expect this to stay similar because the join work is comparable once past the scan.
- `spilled_local_bytes`: data spilled to disk (0 on serverless at this scale).

**What to look for:**
- B should show lower `read_bytes` (less data scanned = less work).
- B should have lower `total_task_duration_ms` (the honest cost signal, not the wall-clock).
- B should show `pruned_files > 0`. The `targetFileSize = '16mb'` setting ensures OPTIMIZE produces 4-5 files instead of 1 or 2. With clustering on `order_date`, each file covers a narrow date band. The sargeable predicate in B skips files whose band falls entirely outside 2024. A (non-sargeable) cannot prune at all. At production scale (hundreds of GB, thousands of files), pruning is even more dramatic.
- The cost ratio from `total_task_duration_ms` is the real answer. Compare it to the wall-clock ratio above. The wall-clock may exaggerate the win because Phase 1 A ran on a colder cache than Phase 2 B.


<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Next</div>
  <div class="ds-callout-title">Deriving DBU per run from system_vw.billing.usage</div>
  <ul>
    <li>Below we query <code>system_vw.billing.usage</code> for billing records that overlap the full benchmark window (Phase 1 start to Phase 2 end), then prorates total DBUs by the fraction of time the benchmark was active.</li>
    <li>We split the estimated cost between A and B proportionally by their median run times.</li>
    <li><strong>Ingestion lag:</strong> <code>system_vw.billing.usage</code> has a 1 to 2 hour delay.</li>
  </ul>
</div>
</div>

In [ ]:
derive_dbu_per_run()

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.info { --accent:#1B5162; --tint:#F8F9FC; }
</style>

<div class="ds-callout info">
  <div class="ds-eyebrow">Tip</div>
  <ul>
    <li>If compute restarts (or you return tomorrow), paste timestamps and medians into the cell below and re-run <code>prove_benchmark()</code> / <code>derive_dbu_per_run()</code>. The query history and billing data live permanently in <code>system_vw.query.history</code> and <code>system_vw.billing.usage</code>.</li>
    <li><code>phase1_start</code> / <code>phase1_end</code>: the two timestamps from <strong>F1a</strong> output, on the line <code>Benchmark window: &lt;start&gt; to &lt;end&gt; UTC</code></li>
    <li><code>phase2_start</code> / <code>phase2_end</code>: the two timestamps from <strong>F1c</strong> output, on the line <code>Benchmark window: &lt;start&gt; to &lt;end&gt; UTC</code></li>
    <li><code>a_median</code>: the seconds value from <strong>F1a</strong> output, on the line <code>A (bad)   median: X.XXs</code></li>
    <li><code>b_median</code>: the seconds value from <strong>F1c</strong> output, on the line <code>Phase 2 B (fixed predicate + clustering): X.XXs</code></li>
  </ul>
</div>
</div>


In [ ]:
# Uncomment and fill in timestamps from the printed output above to re-run after compute restart.
# prove_benchmark(
#     phase1_start="YYYY-MM-DDTHH:MM:SS", phase1_end="YYYY-MM-DDTHH:MM:SS",
#     phase2_start="YYYY-MM-DDTHH:MM:SS", phase2_end="YYYY-MM-DDTHH:MM:SS")

# derive_dbu_per_run(
#     phase1_start="YYYY-MM-DDTHH:MM:SS", phase2_end="YYYY-MM-DDTHH:MM:SS",
#     a_median=0.0, b_median=0.0)


<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: rgba(66,153,224,0.08); --bnr-accent: #1B5162; --bnr-icon-fg: #FFFFFF; --bnr-title: #1B5162; --bnr-tagline: #5E7077; }
/* hero comparison */
.cmp { background: #FFFFFF; border: 1px solid #EEEDE9; border-radius: 10px; padding: 22px 26px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.cmp-row { display: flex; align-items: center; gap: 16px; margin-bottom: 14px; }
.cmp-row:last-child { margin-bottom: 0; }
.cmp-lbl { flex: 0 0 190px; font-size: 14pt; font-weight: 700; color: #0b2026; text-align: right; line-height: 1.3; }
.cmp-lbl small { display: block; font-size: 12px; font-weight: 400; color: #618794; }
.cmp-track { flex: 1; }
.cmp-bar { height: 34px; border-radius: 6px; display: flex; align-items: center; padding: 0 12px; font-size: 13px; font-weight: 700; color: #FFFFFF; white-space: nowrap; }
.cards { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.cards > * { flex: 1; min-width: 280px; }
.card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent); }
.card-title { font-size: 18pt; font-weight: 700; margin: 0 0 10px; color: #0b2026; }
.card ul { margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.card li { margin-bottom: 10px; }
.card li:last-child { margin-bottom: 0; }
.card code { font-family: monospace; font-size: 13pt; background: #EEEDE9; padding: 1px 5px; border-radius: 3px; color: #1B5162; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">&checkmark;</div>
  <div>
    <div class="ds-banner-title">Read cost from task time, not wall-clock</div>
    <div class="ds-banner-tagline">The clock is inflated by the disk cache. Aggregated task time is the honest, cache-neutral cost metric.</div>
  </div>
</div>

<div class="cmp">
  <div class="cmp-row">
    <div class="cmp-lbl">Wall-clock says<small>execution_duration_ms</small></div>
    <div class="cmp-track">
      <div style="display:flex; width:100%;">
        <div class="cmp-bar" style="width:60%; background:#00A972; border-radius:6px 0 0 6px;">B looks much faster</div>
        <div class="cmp-bar" style="width:40%; background:repeating-linear-gradient(45deg,#C4CCD6,#C4CCD6 6px,#EEEDE9 6px,#EEEDE9 12px); color:#5E7077; border-radius:0 6px 6px 0;">cache warmth</div>
      </div>
    </div>
  </div>
  <div class="cmp-row">
    <div class="cmp-lbl">Task time says<small>total_task_duration_ms</small></div>
    <div class="cmp-track">
      <div class="cmp-bar" style="width:60%; background:#4299E0; border-radius:6px;">B is faster (the real work saved)</div>
    </div>
  </div>
  <div style="text-align:center; font-size:13px; color:#618794; margin-top:6px;">The striped slice is the <strong style="color:#98102A;">cache artifact</strong>: wall-clock improvement that is not real work. Task time removes it.</div>
</div>

<div class="cards">
  <div class="card" style="--accent:#4299E0;">
    <div class="card-title">Trust the profile, not the clock</div>
    <ul>
      <li>A runs first on a <strong>colder cache</strong>; B reuses pages A warmed, inflating the wall-clock ratio.</li>
      <li><code>total_task_duration_ms</code> is <strong>cache-neutral</strong>: it sums real compute across all tasks. That gap vs wall-clock is the artifact.</li>
      <li>Lower <code>read_bytes</code> and <code>pruned_files &gt; 0</code> confirm the win is <strong>structural</strong>: fewer files, less data.</li>
    </ul>
  </div>
  <div class="card" style="--accent:#FFAB00;">
    <div class="card-title">Task time = your DBU bill</div>
    <ul>
      <li>On serverless you pay for <strong>aggregated compute ms</strong>. The A / B task-time ratio <strong>is</strong> your cost-savings ratio.</li>
      <li>Faster wall-clock at equal task time = more parallelism, <strong>not less work</strong> and not lower cost.</li>
      <li>Billing confirms after ~2 hours; task time gives the answer <strong>immediately</strong>.</li>
    </ul>
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What each column tells you

Every column in the proof table is the actual `system_vw.query.history` column name (median across 5 runs):

- **`execution_duration_ms`**: wall-clock time per run. Useful for user-perceived latency, but inflated by cache warming for whichever query runs first. Never use this alone for A/B cost comparisons.
- **`total_task_duration_ms`**: aggregated compute milliseconds across all tasks. This is the DBU proxy on serverless. Cache-neutral because it measures actual CPU work, not I/O wait. If A and B have the same value, they cost the same regardless of wall-clock.
- **`read_bytes`** / **`read_files`** / **`pruned_files`**: the scan story. Pruning only fires when the predicate is sargeable AND the file layout clusters data by the filtered column. After clustering, B should show `pruned_files > 0` and lower `read_bytes`.
- **`shuffle_read_bytes`**: data moved between tasks for joins and aggregations. Expect this to stay similar because the join work is comparable once past the scan filter. Large drops here would indicate a broadcast join kicking in (dimension table small enough to broadcast).
- **`spilled_local_bytes`**: data spilled to disk. 0 on serverless at this scale because Photon allocates memory proportional to data volume. On classic compute with fixed executor memory, or at much larger scale, this becomes the leading cost indicator.

### How to read the results

- B shows lower `read_bytes` and `pruned_files > 0`: the `targetFileSize = '16mb'` setting produced 4-5 files, each covering a narrow date band. The sargeable predicate in B skips files whose band falls entirely outside 2024. A (non-sargeable) reads all files.
- At production scale with thousands of files, the same clustering produces even narrower bands and pruning is dramatic.
- B has lower `total_task_duration_ms`: confirms the cost reduction is real, not just a cache artifact. This is the honest DBU comparison.
- Shuffle may vary between phases due to AQE adapting join strategies. Focus on `read_bytes`, `pruned_files`, and `total_task_duration_ms` as the stable, meaningful metrics.
- The wall-clock ratio (from `run_benchmark`) is always larger than the `total_task_duration_ms` ratio (from `prove_benchmark`). The gap is the disk cache artifact. Always report the task duration ratio as the honest cost comparison.

### Why the billing cell may be empty

- `system_vw.billing.usage` has 1 to 2 hours of ingestion lag. In a live session the billing cell will likely return empty.
- Billing records arrive in 10-minute or 60-minute windows. A 1-minute benchmark cannot be exactly matched. The function finds overlapping billing records and prorates DBUs by the fraction of time the benchmark was active.
- The instructor can either (a) re-run the billing cell after the session, or (b) use `total_task_duration_ms` from `system_vw.query.history` as a proxy, which is directly proportional to DBUs on serverless.

### The disk cache artifact (why the clock exaggerates)

- Phase 1 runs A first. The disk cache starts cold. By the time B runs, more data sits in the local SSD cache, making B appear faster even if it does the same work.
- Phase 2 runs only B (after clustering). But Phase 1 A ran on a cold cache while Phase 2 B may benefit from residual warmth.
- `total_task_duration_ms` cuts through this: it measures actual compute regardless of cache state. The task duration ratio is always the honest comparison.
- If you ever see `total_task_duration_ms` identical for A and B but wall-clock very different, the entire "win" is cache warming, not the query fix.

</details>



## Conclusion

In this demo, you built the measurement toolkit that every later part depends on:

1. **Read a Query Profile** end to end: operator tree, Scan (files read vs pruned), Shuffle (bytes moved), and the two clocks (wall-clock vs aggregated task time).
2. **Used Performance Insights** to let the platform rank the bottleneck and name it for you.
3. **Queried `system_vw.query.history`** to compare profile metrics across runs: `read_bytes`, `pruned_files`, `shuffle_read_bytes`, and the cost proxy `total_task_duration_ms`.
4. **Ran an honest two-phase benchmark**: cache-busting comments, warmup discarded, median reported. Phase 1 showed the predicate fix alone does not prune (no clustering). Phase 2 proved the predicate + layout fix together enable pruning.
5. **Proved the win from the profile**: `pruned_files > 0`, lower `read_bytes`, lower `total_task_duration_ms`. The wall-clock exaggerated because of cache warming; `total_task_duration_ms` gave the stable, cache-neutral comparison.

Every later part in this course follows the same approach: measure the current state, identify the bottleneck, change one variable, and confirm the improvement in the profile metrics.



&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>